# Online Retail Intelligence Capstone
## Notebook 01: Data Collection and Validation

### Project Objective
Analyse historical online retail transactions to understand sales
performance, customer purchasing behaviour, and opportunities
for customer segmentation.

### Business Questions
1. How does retail sales performance vary over time?
2. Which products and geographic markets contribute most to revenue?
3. What are the purchasing patterns of retail customers?
4. How can customers be segmented based on purchasing behaviour?
5. Which historical purchasing behaviours are associated with future repeat purchases?

### Notebook Objectives
- Import the required Python libraries.
- Load the Online Retail dataset.
- Examine the dataset's structure and data types.
- Identify missing values and duplicate records.
- Document initial data quality observations.

### Dataset Source
The dataset was obtained from Kaggle:

**Dataset:** Online Retail Transactions Dataset

**Source:** https://www.kaggle.com/datasets/abhishekrp1517/online-retail-transactions-dataset

In [29]:
# Import Pandas for data loading, manipulation, and analysis.
import pandas as pd

# Import NumPy for numerical calculations and array operations.
import numpy as np

# Import Matplotlib for creating statistical charts and visualisations.
import matplotlib.pyplot as plt

# Import Seaborn for enhanced statistical data visualisation.
import seaborn as sns

# Import Path to manage file and directory paths across operating systems.
from pathlib import Path

# Apply a consistent white-grid theme to improve chart readability.
sns.set_theme(style="whitegrid")

# Confirm that the required libraries have been imported successfully.
print("Libraries imported successfully!")

Libraries imported successfully!


In [30]:
# Import KaggleHub to download datasets programmatically from Kaggle.
import kagglehub

# Define the Kaggle dataset identifier.
# This identifies the Online Retail Transactions Dataset hosted on Kaggle.
dataset_id = "abhishekrp1517/online-retail-transactions-dataset"

# Download the latest available version of the dataset.
# KaggleHub stores the downloaded files in a local cache directory.
# Path converts the returned location into a Path object.
dataset_path = Path(
    kagglehub.dataset_download(dataset_id)
)

# Display the directory containing the downloaded dataset.
# This helps verify where KaggleHub has stored the files.
print("Dataset downloaded to:", dataset_path)

# Display the filenames available in the downloaded directory.
# This allows us to identify the correct CSV file before loading it.
print("\nAvailable files:")

# Iterate through the directory and print each filename.
for file in dataset_path.iterdir():
    print(file.name)

Dataset downloaded to: C:\Users\Joy\.cache\kagglehub\datasets\abhishekrp1517\online-retail-transactions-dataset\versions\2

Available files:
Online Retail.csv
Online Retail.xlsx


In [31]:
# STEP 1: Locate CSV files in the downloaded Kaggle dataset.
# rglob() searches the dataset directory and its subdirectories.
csv_files = list(dataset_path.rglob("*.csv"))

# STEP 2: Display the available CSV files.
# This helps verify the filename before loading the data.
print("Available CSV files:")

for file in csv_files:
    print(file.name)

# STEP 3: Confirm that exactly one CSV file was found.
# This prevents accidentally selecting the wrong dataset.
if len(csv_files) != 1:
    raise ValueError(
        f"Expected one CSV file, but found {len(csv_files)}."
    )

# STEP 4: Load the CSV into a Pandas DataFrame.
# ISO-8859-1 supports special characters in product descriptions.
df = pd.read_csv(
    csv_files[0],
    encoding="ISO-8859-1"
)

# STEP 5: Display the dataset dimensions.
# The shape attribute returns (number of rows, number of columns).
print("\nDataset shape:", df.shape)

# STEP 6: Preview the first five records.
# This helps us understand the dataset's columns and values.
display(df.head())

Available CSV files:
Online Retail.csv

Dataset shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


## Initial Data Quality Assessment

This section examines the structure and quality of the Kaggle Online Retail dataset before preprocessing.

The assessment covers:
- Dataset dimensions and column data types
- Missing values
- Duplicate records
- Invalid quantities and unit prices
- Transaction date range

The original dataset will remain unchanged during this assessment.

In [32]:
# STEP 1: Display the dataset's structure and column data types.
# This helps identify fields that may require type conversion.
print("DATASET INFORMATION")
df.info()

# STEP 2: Calculate missing values for each column.
# Missing values may affect customer-level analysis and reporting.
print("\nMISSING VALUES")
missing_values = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
})
display(missing_values)

# STEP 3: Count exact duplicate rows.
# Duplicates require investigation before removal.
duplicate_count = df.duplicated().sum()
print("\nExact duplicate rows:", duplicate_count)

# STEP 4: Identify non-positive quantities and unit prices.
# These records may represent cancellations, returns, or invalid entries.
print("\nNON-POSITIVE VALUES")
print("Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("UnitPrice <= 0:", (df["UnitPrice"] <= 0).sum())

# STEP 5: Identify invoices marked as cancellations.
# Invoice numbers beginning with 'C' may indicate cancelled transactions.
cancellation_count = (
    df["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
    .sum()
)
print("\nCancellation-related rows:", cancellation_count)

# STEP 6: Convert invoice dates for validation.
# errors='coerce' converts invalid date strings into missing datetime values.
invoice_dates = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

# STEP 7: Report the transaction period and invalid dates.
print("\nTRANSACTION DATE RANGE")
print("Earliest transaction:", invoice_dates.min())
print("Latest transaction:", invoice_dates.max())
print("Invalid dates:", invoice_dates.isna().sum())

DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   541909 non-null  int64  
 7   Country      541909 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 33.1+ MB

MISSING VALUES


,Missing Count,Missing Percentage
InvoiceNo,0,0.00
StockCode,0,0.00
Description,1454,0.27
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,0,0.00
Country,0,0.00



Exact duplicate rows: 5268

NON-POSITIVE VALUES
Quantity <= 0: 10624
UnitPrice <= 0: 2517

Cancellation-related rows: 9288

TRANSACTION DATE RANGE
Earliest transaction: 2010-12-01 08:26:00
Latest transaction: 2011-12-09 12:50:00
Invalid dates: 0


### Initial Data Quality Findings

The Kaggle Online Retail dataset contains 541,909 transaction
records and eight columns, covering 1 December 2010 to
9 December 2011.

The initial assessment identified the following:

- **Missing descriptions:** 1,454 records (0.27%).
- **Exact duplicates:** 5,268 records.
- **Non-positive quantities:** 10,624 records.
- **Non-positive unit prices:** 2,517 records.
- **Cancellation-related records:** 9,288 rows.
- **Invalid invoice dates:** None.
- **Missing customer IDs:** None detected.

### Implications for Analysis

Non-positive quantities and cancellation records may represent
returns or adjustments rather than completed sales.

Duplicate transactions require investigation before removal.

Missing product descriptions may affect product-level analysis.

Customer identifiers must be examined for potential placeholder
values before customer segmentation.

### Proposed Cleaning Strategy

1. Preserve the original dataset for auditability.
2. Convert InvoiceDate to datetime.
3. Investigate duplicate records.
4. Separate valid sales from cancellations and returns.
5. Exclude non-positive prices from completed-sales analysis.
6. Investigate missing product descriptions.
7. Validate customer identifiers before RFM analysis.
8. Document all cleaning decisions and resulting record counts.

In [33]:
# STEP 1: Count the number of unique customer identifiers.
# This provides an initial measure of the customer population.
unique_customers = df["CustomerID"].nunique()

print("Unique customer IDs:", unique_customers)

# STEP 2: Identify the ten most frequently occurring customer IDs.
# An unusually frequent ID may represent a placeholder or a large buyer.
customer_frequency = df["CustomerID"].value_counts().head(10)

print("\nTop 10 customer IDs by transaction rows:")
display(customer_frequency.to_frame(name="Transaction Rows"))

# STEP 3: Calculate each customer's share of all transaction rows.
# A very high concentration may require further investigation.
top_customer_id = customer_frequency.index[0]
top_customer_count = customer_frequency.iloc[0]

top_customer_percentage = (
    top_customer_count / len(df)
) * 100

print("\nMost frequent customer ID:", top_customer_id)
print("Transaction rows:", top_customer_count)
print(
    "Percentage of dataset:",
    round(top_customer_percentage, 2),
    "%"
)

Unique customer IDs: 4372

Top 10 customer IDs by transaction rows:


,Transaction Rows
CustomerID,
15287,135101
17841,7983
14911,5903
14096,5128
12748,4642
14606,2782
15311,2491
14646,2085
13089,1857



Most frequent customer ID: 15287
Transaction rows: 135101
Percentage of dataset: 24.93 %


In [34]:
# STEP 1: Isolate records associated with CustomerID 15287.
# This customer ID accounts for an unusually large share of rows.
customer_15287 = df[df["CustomerID"] == 15287].copy()

# STEP 2: Count distinct invoices.
# Invoice counts are more meaningful than transaction-line counts
# when investigating purchasing frequency.
print("Distinct invoices:",
      customer_15287["InvoiceNo"].nunique())

# STEP 3: Examine the countries associated with this ID.
# This may help identify whether the records represent one customer.
print("\nCountries:")
display(customer_15287["Country"].value_counts())

# STEP 4: Examine the earliest and latest transaction dates.
# This helps establish the period covered by the records.
dates_15287 = pd.to_datetime(
    customer_15287["InvoiceDate"],
    errors="coerce"
)

print("\nFirst transaction:", dates_15287.min())
print("Last transaction:", dates_15287.max())

# STEP 5: Check how many rows represent cancellations.
# This provides additional context about the transaction history.
cancellations_15287 = (
    customer_15287["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
    .sum()
)

print("\nCancellation-related rows:", cancellations_15287)

# STEP 6: Preview sample records.
# These examples help us inspect the ID before deciding what to do.
display(customer_15287.head(10))

Distinct invoices: 3713

Countries:


Country
United Kingdom    133621
EIRE                 711
Hong Kong            288
Unspecified          202
Switzerland          125
France                66
Israel                47
Portugal              39
Bahrain                2
Name: count, dtype: int64


First transaction: 2010-12-01 11:52:00
Last transaction: 2011-12-09 10:26:00

Cancellation-related rows: 384


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,15287,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,15287,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,15287,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,15287,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,15287,United Kingdom
1447,536544,21790,VINTAGE SNAP CARDS,9,2010-12-01 14:32:00,1.66,15287,United Kingdom
1448,536544,21791,VINTAGE HEADS AND TAILS CARD GAME,2,2010-12-01 14:32:00,2.51,15287,United Kingdom
1449,536544,21801,CHRISTMAS TREE DECORATION WITH BELL,10,2010-12-01 14:32:00,0.43,15287,United Kingdom
1450,536544,21802,CHRISTMAS TREE HEART DECORATION,9,2010-12-01 14:32:00,0.43,15287,United Kingdom
1451,536544,21803,CHRISTMAS TREE STAR DECORATION,11,2010-12-01 14:32:00,0.43,15287,United Kingdom


### CustomerID 15287: Investigation Findings

CustomerID 15287 accounts for 135,101 transaction rows (24.93%
of the dataset), covering 3,713 distinct invoices across nine
countries.

The identifier appears throughout the transaction period and
includes 384 cancellation-related rows.

Its unusually high frequency and geographical distribution
raise concerns that it may represent a shared or substituted
customer identifier rather than one individual customer.

**Analytical decision:**
- Preserve the records in the original dataset.
- Flag CustomerID 15287 for further validation.
- Provisionally exclude it from the primary RFM segmentation.
- Retain eligible transactions for aggregate sales analysis.
- Document the potential impact on customer-level findings.

This decision will be reviewed during data cleaning and
customer segmentation.